In [1]:
import nltk
import re
from nltk.wsd import lesk
from nltk.corpus import wordnet

In [2]:
nltk.download('all', quiet=True)

True

In [3]:
ENTITIES = {
    "PATIENT": r"\b(patient|man|woman|child|infant)\b",
    "CONDITION": r"\b(fever|infection|wound|pain)\b",
    "LOCATION": r"\b(hospital|icu|ward|clinic|lab|laboratory)\b"
}

In [4]:
def analyze_domain_lesk(text):
  tokens = re.findall(r"\b\w+\b", text.lower())
  frame = {}
  for key, pattern in ENTITIES.items():
    match = re.search(pattern, text, re.IGNORECASE)
    frame[key] = match.group(0) if match else "UNKNOWN"

  action_words = [
      "discharge", "admit", "admitted", "detect", "detected", "treat", "treated", "culture"
  ]

  frame["ACTION"] = next(
      (tokens for token in tokens if token in action_words),
      "UNKNOWN"
  )

  target_words = ["discharge", "culture"]

  frame["SENSES"] = {}

  for word in tokens:
    if word in target_words:
      synset = lesk(tokens, word)
      if word == "discharge" and "will" in tokens :
        pos = "v"
      else:
        pos = "n"

      pos_synset = lesk(tokens, word, pos=pos)

      final_synset = pos_synset if pos_synset else synset

      if final_synset:
                frame["SENSES"][word] = (
                    final_synset.name(),
                    final_synset.definition()
                )
      else:
                frame["SENSES"][word] = (
                    "UNKNOWN",
                    "No sense found"
                )

    return frame


# Test statements
statements = [
    "The doctor will discharge the patient from the hospital ward today.",
    "The laboratory detected a bacterial culture in the wound discharge.",
    "The infant was detected with bacterial infection."
]


# Analyze each statement
for i, stmt in enumerate(statements, 1):
    res = analyze_domain_lesk(stmt)

    print(f'\n[{i}] "{stmt}"')

    print(
        f"Frame : "
        f"Patient={res['PATIENT']} | "
        f"Action={res['ACTION']} | "
        f"Condition={res['CONDITION']} | "
        f"Location={res['LOCATION']}"
    )

    for word, (syn_name, definition) in res["SENSES"].items():
        print(
            f"NLTK Lesk : '{word}' -> "
            f"{syn_name} : {definition}"
        )




[1] "The doctor will discharge the patient from the hospital ward today."
Frame : Patient=patient | Action=['the', 'doctor', 'will', 'discharge', 'the', 'patient', 'from', 'the', 'hospital', 'ward', 'today'] | Condition=UNKNOWN | Location=hospital

[2] "The laboratory detected a bacterial culture in the wound discharge."
Frame : Patient=UNKNOWN | Action=['the', 'laboratory', 'detected', 'a', 'bacterial', 'culture', 'in', 'the', 'wound', 'discharge'] | Condition=wound | Location=laboratory

[3] "The infant was detected with bacterial infection."
Frame : Patient=infant | Action=['the', 'infant', 'was', 'detected', 'with', 'bacterial', 'infection'] | Condition=infection | Location=UNKNOWN
